# AlgoTrader Evo: Experiment Analysis

Analysis of autonomous strategy evolution results from `results.tsv`.

**Key Metrics:**
- **Score:** Composite metric (Sharpe + Calmar + Robustness). Higher is better.
- **Drawdown:** Maximum drawdown observed in OOS testing.
- **Status:** KEEP (Improved), DISCARD (Worse), CRASH (Error).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)

# Load the TSV 
try:
    df = pd.read_csv("results.tsv", sep="\t")
except FileNotFoundError:
    print("results.tsv not found. Run the agent loop first.")
    df = pd.DataFrame()

if not df.empty:
    # Clean data
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["drawdown"] = pd.to_numeric(df.get("drawdown", pd.Series([0]*len(df))), errors="coerce")
    df["trades"] = pd.to_numeric(df.get("trades", pd.Series([0]*len(df))), errors="coerce")
    df["status"] = df["status"].str.strip().str.upper()

    print(f"Total experiments: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(10))
else:
    print("No data to analyze yet.")

In [ ]:
if not df.empty:
    counts = df["status"].value_counts()
    print("Experiment Outcomes:")
    print(counts.to_string())

    n_keep = counts.get("KEEP", 0)
    n_discard = counts.get("DISCARD", 0)
    n_crash = counts.get("CRASH", 0)
    n_decided = n_keep + n_discard
    
    if n_decided > 0:
        keep_rate = n_keep / n_decided
        print(f"\nKeep Rate: {n_keep}/{n_decided} = {keep_rate:.1%}")
    
    # Show crash reasons if available
    if n_crash > 0:
        crashes = df[df["status"] == "CRASH"]
        print(f"\nTop Crash Descriptions:")
        print(crashes["description"].head(5).to_string())

## Kept Strategies (The Evolution Frontier)

Listing all strategies that improved upon the previous best.

In [ ]:
if not df.empty:
    kept = df[df["status"] == "KEEP"].copy()
    print(f"KEPT experiments ({len(kept)} total):\n")
    
    if len(kept) > 0:
        # Format output table
        display_cols = ["score", "drawdown", "trades", "description"]
        for col in display_cols:
            if col not in kept.columns:
                kept[col] = None
                
        kept["drawdown"] = kept["drawdown"].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else "N/A")
        kept["score"] = kept["score"].round(4)
        
        display(kept[["score", "drawdown", "trades", "description"]])
    else:
        print("No successful improvements found yet.")

## Equity Curve Evolution

Tracks how the best **Score** evolves. The running maximum shows the performance frontier.

In [ ]:
if not df.empty and len(df) > 1:
    fig, ax = plt.subplots(figsize=(16, 8))

    valid = df[df["status"] != "CRASH"].copy()
    valid = valid.reset_index(drop=True)

    baseline_score = valid.loc[0, "score"]
    best_score = valid["score"].max()

    disc = valid[valid["status"] == "DISCARD"]
    ax.scatter(disc.index, disc["score"],
               c="#cccccc", s=15, alpha=0.4, zorder=2, label="Discarded")

    kept_v = valid[valid["status"] == "KEEP"]
    ax.scatter(kept_v.index, kept_v["score"],
               c="#2ecc71", s=80, zorder=4, label="Kept (Improvement)", 
               edgecolors="black", linewidths=1)

    kept_mask = valid["status"] == "KEEP"
    kept_idx = valid.index[kept_mask]
    kept_scores = valid.loc[kept_mask, "score"]
    running_max = kept_scores.cummax()
    
    ax.step(kept_idx, running_max, where="post", color="#27ae60",
            linewidth=3, alpha=0.8, zorder=3, label="Best So Far")

    n_total = len(df)
    n_kept = len(df[df["status"] == "KEEP"])
    
    ax.set_xlabel("Experiment Iteration #", fontsize=12, fontweight='bold')
    ax.set_ylabel("Composite Score", fontsize=12, fontweight='bold')
    ax.set_title(f"AlgoTrader Evo Progress: {n_total} Runs, {n_kept} Improvements\nBaseline: {baseline_score:.3f} -> Best: {best_score:.3f}", 
                 fontsize=14, pad=20)
    
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.3, label="Zero Score")
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("evolution_progress.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved plot to evolution_progress.png")
else:
    print("Not enough data to plot evolution.")

## Risk vs. Reward Scatter

Visualizing the trade-off between **Score** (Return) and **Drawdown** (Risk).
*Ideal strategies are in the Top-Left (High Score, Low Drawdown).*

In [ ]:
if not df.empty and "drawdown" in df.columns:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    plot_dd = df["drawdown"].abs()
    colors = df["status"].map({"KEEP": "#2ecc71", "DISCARD": "#95a5a6", "CRASH": "#e74c3c"})
    
    ax.scatter(plot_dd, df["score"], 
               c=colors, s=60, alpha=0.7, edgecolors='k', linewidth=0.5)
    
    if len(df[df["status"]=="KEEP"]) > 0:
        best_idx = df[df["status"]=="KEEP"]["score"].idxmax()
        best_row = df.loc[best_idx]
        ax.scatter(abs(best_row["drawdown"]), best_row["score"], 
                   c="#f1c40f", s=200, marker='*', edgecolors='black', linewidth=2, 
                   label=f"Best Strategy (Score: {best_row['score']:.2f})")

    ax.set_xlabel("Max Drawdown (Lower is Better)", fontsize=12)
    ax.set_ylabel("Composite Score (Higher is Better)", fontsize=12)
    ax.set_title("Risk vs. Reward: All Experiments", fontsize=14)
    
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("risk_reward_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Drawdown data not available for risk/reward plot.")

## Final Summary Statistics

In [ ]:
if not df.empty:
    kept = df[df["status"] == "KEEP"].copy()
    
    if len(kept) > 0:
        baseline_score = df.iloc[0]["score"]
        best_score = kept["score"].max()
        best_row = kept.loc[kept["score"].idxmax()]
        
        improvement = best_score - baseline_score
        pct_imp = (improvement / abs(baseline_score)) * 100 if baseline_score != 0 else 0
        
        print(f"Performance Summary:")
        print(f"   Baseline Score:  {baseline_score:.4f}")
        print(f"   Best Score:      {best_score:.4f}")
        print(f"   Total Improvement: +{improvement:.4f} ({pct_imp:.1f}%)")
        print(f"\nBest Strategy Details:")
        print(f"   Description: {best_row['description']}")
        print(f"   Drawdown:    {best_row.get('drawdown', 'N/A')}")
        print(f"   Trades:      {best_row.get('trades', 'N/A')}")
    else:
        print("No kept strategies found yet. The agent is still searching.")